# NB01 — Data Processing
**Flight Delay Prediction Using Machine Learning**

This notebook loads 2018–2019 BTS domestic US flight data from parquet files, merges airport coordinates, fetches hourly weather from Open-Meteo, and produces a clean CSV ready for EDA and feature engineering.

**Outputs:** `01_cleaned_data.csv`

In [79]:
import pandas as pd
import numpy as np
import requests
import time
import os

# ---- Paths (adjust to your setup) ----
PARQUET_2018 = r"C:\YEAR 3 - FIRST SEM\flight-delay-predictor-6\Data\Combined_Flights_2018.parquet"
PARQUET_2019 = r"C:\YEAR 3 - FIRST SEM\flight-delay-predictor-6\Data\Combined_Flights_2019.parquet"
AIRPORTS_PATH = r"C:\YEAR 3 - FIRST SEM\flight-delay-predictor-6\Data\airports.csv"
WEATHER_CACHE_PATH = r"C:\YEAR 3 - FIRST SEM\flight-delay-predictor-6\Data\weather_cache"
OUTPUT_PATH = r"C:\YEAR 3 - FIRST SEM\flight-delay-predictor-6\Data\01_cleaned_data.csv"

os.makedirs(WEATHER_CACHE_PATH, exist_ok=True)
print('Paths set.')

Paths set.


---
## 1 — Load & Merge Parquets

In [98]:
print('Loading parquets...')
df_2018 = pd.read_parquet(PARQUET_2018)
df_2019 = pd.read_parquet(PARQUET_2019)

print(f'2018: {df_2018.shape[0]:,} rows × {df_2018.shape[1]} cols')
print(f'2019: {df_2019.shape[0]:,} rows × {df_2019.shape[1]} cols')

# Confirm columns match
assert list(df_2018.columns) == list(df_2019.columns), 'Column mismatch!'
print(f'\nColumns match: {len(df_2018.columns)} columns')

df = pd.concat([df_2018, df_2019], ignore_index=True)
del df_2018, df_2019  # free memory

print(f'\nCombined: {df.shape[0]:,} rows')
print(f'\nColumns:\n{df.columns.tolist()}')

Loading parquets...
2018: 5,689,512 rows × 61 cols
2019: 8,091,684 rows × 61 cols

Columns match: 61 columns

Combined: 13,781,196 rows

Columns:
['FlightDate', 'Airline', 'Origin', 'Dest', 'Cancelled', 'Diverted', 'CRSDepTime', 'DepTime', 'DepDelayMinutes', 'DepDelay', 'ArrTime', 'ArrDelayMinutes', 'AirTime', 'CRSElapsedTime', 'ActualElapsedTime', 'Distance', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'Marketing_Airline_Network', 'Operated_or_Branded_Code_Share_Partners', 'DOT_ID_Marketing_Airline', 'IATA_Code_Marketing_Airline', 'Flight_Number_Marketing_Airline', 'Operating_Airline', 'DOT_ID_Operating_Airline', 'IATA_Code_Operating_Airline', 'Tail_Number', 'Flight_Number_Operating_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'DepDel1

In [99]:
print(f'Total columns: {len(df.columns)}')
print(f'Total rows: {len(df):,}')
print(f'\n=== All Columns ===')
for i, col in enumerate(df.columns):
    dtype = df[col].dtype
    nunique = df[col].nunique()
    nulls = df[col].isna().sum()
    sample = df[col].dropna().iloc[0] if nulls < len(df) else 'ALL NULL'
    print(f'{i+1:>3}. {col:<45} dtype={str(dtype):<15} unique={nunique:<10} nulls={nulls:<10} sample={sample}')

Total columns: 61
Total rows: 13,781,196

=== All Columns ===
  1. FlightDate                                    dtype=datetime64[us]  unique=730        nulls=0          sample=2018-01-23 00:00:00
  2. Airline                                       dtype=str             unique=28         nulls=0          sample=Endeavor Air Inc.
  3. Origin                                        dtype=str             unique=375        nulls=0          sample=ABY
  4. Dest                                          dtype=str             unique=375        nulls=0          sample=ATL
  5. Cancelled                                     dtype=bool            unique=2          nulls=0          sample=False
  6. Diverted                                      dtype=bool            unique=2          nulls=0          sample=False
  7. CRSDepTime                                    dtype=int64           unique=1411       nulls=0          sample=1202
  8. DepTime                                       dtype=float64      

In [100]:
df.head(3)

,FlightDate,Airline,Origin,Dest,Cancelled,Diverted,CRSDepTime,DepTime,DepDelayMinutes,DepDelay,...,WheelsOff,WheelsOn,TaxiIn,CRSArrTime,ArrDelay,ArrDel15,ArrivalDelayGroups,ArrTimeBlk,DistanceGroup,DivAirportLandings
0,2018-01-23,Endeavor Air Inc.,ABY,ATL,False,False,1202,1157.0,0.0,-5.0,...,1211.0,1249.0,7.0,1304,-8.0,0.0,-1.0,1300-1359,1,0.0
1,2018-01-24,Endeavor Air Inc.,ABY,ATL,False,False,1202,1157.0,0.0,-5.0,...,1210.0,1246.0,12.0,1304,-6.0,0.0,-1.0,1300-1359,1,0.0
2,2018-01-25,Endeavor Air Inc.,ABY,ATL,False,False,1202,1153.0,0.0,-9.0,...,1211.0,1251.0,11.0,1304,-2.0,0.0,-1.0,1300-1359,1,0.0


---
## 2 — Initial Cleaning

In [104]:
# ---- Standardise column names ----
# Map the raw BTS columns to our working names
df = df.rename(columns={
    'FlightDate': 'DATE',
    'Operating_Airline': 'AIRLINE',
    'Origin': 'ORIGIN_AIRPORT',
    'Dest': 'DESTINATION_AIRPORT',
    'CRSDepTime': 'SCHEDULED_DEPARTURE',
    'DepDelay': 'DEPARTURE_DELAY',
    'DepDelayMinutes': 'DEP_DELAY_MINUTES',
    'Distance': 'DISTANCE',
    'Tail_Number': 'TAIL_NUMBER',
    'CRSArrTime': 'SCHEDULED_ARRIVAL',
    'CRSElapsedTime': 'SCHEDULED_ELAPSED_TIME',
    'DepDel15': 'DEP_DEL_15',
    'DayOfWeek': 'DAY_OF_WEEK',
    'DayofMonth': 'DAY_OF_MONTH',
})

df['DATE'] = pd.to_datetime(df['DATE'])
df['YEAR'] = df['DATE'].dt.year

print(f'Years present: {sorted(df["YEAR"].unique())}')
print(df['YEAR'].value_counts().sort_index())

Years present: [np.int32(2018), np.int32(2019)]
YEAR
2018    5586081
2019    7917264
Name: count, dtype: int64


In [ ]:
print('Loading and filtering parquets...')

# Only keep the columns we need to limit memory before any further processing.
# The raw parquets have 61 columns each, most of which are post-departure data
# (taxi times, wheels-off / wheels-on) that would leak into the prediction target.
keep_cols = ['FlightDate', 'Operating_Airline', 'Origin', 'Dest', 'Cancelled', 'Diverted',
             'CRSDepTime', 'DepDelay', 'Distance', 'Tail_Number', 'CRSArrTime',
             'CRSElapsedTime', 'DayOfWeek', 'ArrDelay']

df_2018 = pd.read_parquet(PARQUET_2018, columns=keep_cols)
df_2019 = pd.read_parquet(PARQUET_2019, columns=keep_cols)

# Drop cancelled / diverted flights (no meaningful delay) and rows with no
# DepDelay value. Done before concat so the intermediate frames stay smaller.
df_2018 = df_2018[(df_2018['Cancelled'] == 0) & (df_2018['Diverted'] == 0)].drop(columns=['Cancelled', 'Diverted'])
df_2019 = df_2019[(df_2019['Cancelled'] == 0) & (df_2019['Diverted'] == 0)].drop(columns=['Cancelled', 'Diverted'])

df_2018 = df_2018.dropna(subset=['DepDelay'])
df_2019 = df_2019.dropna(subset=['DepDelay'])

print(f'2018 after filter: {len(df_2018):,}')
print(f'2019 after filter: {len(df_2019):,}')

df = pd.concat([df_2018, df_2019], ignore_index=True)
del df_2018, df_2019  # free the originals; we only need the merged frame from here on

print(f'Combined: {len(df):,} rows × {len(df.columns)} cols')

In [106]:
df = df.rename(columns={
    'FlightDate': 'DATE',
    'Operating_Airline': 'AIRLINE',
    'Origin': 'ORIGIN_AIRPORT',
    'Dest': 'DESTINATION_AIRPORT',
    'CRSDepTime': 'SCHEDULED_DEPARTURE',
    'DepDelay': 'DEPARTURE_DELAY',
    'Distance': 'DISTANCE',
    'Tail_Number': 'TAIL_NUMBER',
    'CRSArrTime': 'SCHEDULED_ARRIVAL',
    'CRSElapsedTime': 'SCHEDULED_ELAPSED_TIME',
    'DayOfWeek': 'DAY_OF_WEEK',
})

df['DATE'] = pd.to_datetime(df['DATE'])
df['YEAR'] = df['DATE'].dt.year

print(f'Columns: {df.columns.tolist()}')

Columns: ['DATE', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_DELAY', 'DISTANCE', 'TAIL_NUMBER', 'SCHEDULED_ARRIVAL', 'SCHEDULED_ELAPSED_TIME', 'DAY_OF_WEEK', 'ArrDelay', 'YEAR']


In [ ]:
# Derive HOUR from CRSDepTime which is stored as HHMM (e.g. 1430 → 14).
# Clipped to 0–23 to handle any malformed values.
df['HOUR'] = (df['SCHEDULED_DEPARTURE'] // 100).astype(int).clip(0, 23)

# Binary delay target: 1 if delayed by 15 minutes or more, otherwise 0.
# 15-minute cutoff aligns with the BTS / FAA convention for "delayed".
df['TARGET'] = (df['DEPARTURE_DELAY'] >= 15).astype(int)

print(f'Class balance:')
print(df['TARGET'].value_counts(normalize=True).round(3))
print(f'\nDate range: {df["DATE"].min().date()} to {df["DATE"].max().date()}')
print(f'Total rows: {len(df):,}')

---
## 3 — Merge Airport Coordinates

In [108]:
airports = pd.read_csv(AIRPORTS_PATH)
print(f'Airports CSV: {airports.shape[0]} rows')
print(f'Columns: {airports.columns.tolist()}')
airports.head(3)

Airports CSV: 322 rows
Columns: ['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'COUNTRY', 'LATITUDE', 'LONGITUDE']


,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919


In [109]:
# Merge origin airport coordinates
df = df.merge(
    airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']],
    left_on='ORIGIN_AIRPORT',
    right_on='IATA_CODE',
    how='left'
).drop(columns='IATA_CODE')

missing = df['LATITUDE'].isna().sum()
print(f'Rows missing coordinates: {missing:,} ({missing/len(df)*100:.1f}%)')
df = df.dropna(subset=['LATITUDE'])
print(f'Rows after coord merge: {len(df):,}')

Rows missing coordinates: 152,622 (1.1%)
Rows after coord merge: 13,350,723


---
## 4 — Fetch Hourly Weather from Open-Meteo
Uses the Archive API (ERA5 reanalysis). Fetches once, caches to disk. Run Cell 4a to fetch, then Cell 4b to load from cache on subsequent runs.

In [43]:
unique_airports = df[['ORIGIN_AIRPORT', 'LATITUDE', 'LONGITUDE']].drop_duplicates()
print(f'Unique origin airports needing weather: {len(unique_airports)}')

Unique origin airports needing weather: 316


In [44]:
WEATHER_VARS = "temperature_2m,precipitation,snowfall,wind_speed_10m,wind_gusts_10m,weather_code"

In [ ]:
import time

# Fetch hourly weather for every unique origin airport over the full 2018-2019
# window. One request per airport returns ~17,500 rows (24 hours × 730 days).
# Open-Meteo rate-limits aggressive callers, so we sleep 6s between calls.
unique_airports = df[['ORIGIN_AIRPORT', 'LATITUDE', 'LONGITUDE']].drop_duplicates()
print(f"Fetching weather for {len(unique_airports)} airports...")

weather_records = []
failed = []  # rate-limited or networking failures, retried in the next cell

for i, (_, row) in enumerate(unique_airports.iterrows()):
    try:
        params = {
            "latitude": row['LATITUDE'],
            "longitude": row['LONGITUDE'],
            "start_date": "2018-01-01",
            "end_date": "2019-12-31",
            "hourly": "temperature_2m,precipitation,snowfall,wind_speed_10m,wind_gusts_10m,weather_code",
            "wind_speed_unit": "mph",
            "timezone": "UTC"
        }

        response = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params=params,
            timeout=30
        )

        if response.status_code != 200:
            failed.append(row['ORIGIN_AIRPORT'])
            print(f"HTTP {response.status_code} for {row['ORIGIN_AIRPORT']}")
            continue

        # Open-Meteo returns parallel arrays under data['hourly']; one entry per hour.
        data = response.json()
        hourly = data['hourly']

        weather_df = pd.DataFrame({
            'DATETIME': pd.to_datetime(hourly['time']),
            'ORIGIN_AIRPORT': row['ORIGIN_AIRPORT'],
            'temp': hourly['temperature_2m'],
            'precip': hourly['precipitation'],
            'snowfall': hourly['snowfall'],
            'wind_speed': hourly['wind_speed_10m'],
            'wind_gusts': hourly['wind_gusts_10m'],
            'weather_code': hourly['weather_code'],
        })

        weather_records.append(weather_df)

        if (i + 1) % 25 == 0:
            print(f"  {i + 1} airports done...")

        time.sleep(6)  # politeness delay; tightening this triggers HTTP 429

    except Exception as e:
        failed.append(row['ORIGIN_AIRPORT'])
        print(f"Failed: {row['ORIGIN_AIRPORT']} — {e}")

# Split DATETIME into DATE + HOUR so we can join with the flight frame on the
# same keys used downstream.
weather_all = pd.concat(weather_records, ignore_index=True)
weather_all['DATE'] = pd.to_datetime(weather_all['DATETIME'].dt.date)
weather_all['HOUR'] = weather_all['DATETIME'].dt.hour
weather_all = weather_all.drop(columns='DATETIME')

# Cache to disk so subsequent notebook runs skip the API entirely.
weather_all.to_parquet(os.path.join(WEATHER_CACHE_PATH, 'hourly_weather_2018_2019.parquet'), index=False)
print(f"\nDone. Success: {len(weather_records)}, Failed: {len(failed)}")
if failed:
    print(f"Failed airports: {failed}")

In [ ]:
# Retry pass for airports that hit HTTP 429 in the initial fetch.
# Same logic as cell above but only over the failed list. Successes are
# concatenated onto the cached parquet so the file stays the single source of truth.
retry_airports = unique_airports[unique_airports['ORIGIN_AIRPORT'].isin(failed)]
print(f'Retrying {len(retry_airports)} airports...')

retry_records = []
still_failed = []

for _, row in retry_airports.iterrows():
    try:
        params = {
            "latitude": row['LATITUDE'],
            "longitude": row['LONGITUDE'],
            "start_date": "2018-01-01",
            "end_date": "2019-12-31",
            "hourly": "temperature_2m,precipitation,snowfall,wind_speed_10m,wind_gusts_10m,weather_code",
            "wind_speed_unit": "mph",
            "timezone": "UTC"
        }
        response = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=30)
        if response.status_code != 200:
            still_failed.append(row['ORIGIN_AIRPORT'])
            continue
        data = response.json()
        hourly = data['hourly']
        retry_records.append(pd.DataFrame({
            'DATETIME': pd.to_datetime(hourly['time']),
            'ORIGIN_AIRPORT': row['ORIGIN_AIRPORT'],
            'temp': hourly['temperature_2m'],
            'precip': hourly['precipitation'],
            'snowfall': hourly['snowfall'],
            'wind_speed': hourly['wind_speed_10m'],
            'wind_gusts': hourly['wind_gusts_10m'],
            'weather_code': hourly['weather_code'],
        }))
        time.sleep(6)
    except:
        still_failed.append(row['ORIGIN_AIRPORT'])

if retry_records:
    new = pd.concat(retry_records, ignore_index=True)
    new['DATE'] = pd.to_datetime(new['DATETIME'].dt.date)
    new['HOUR'] = new['DATETIME'].dt.hour
    new = new.drop(columns='DATETIME')
    combined = pd.concat([weather_all, new], ignore_index=True)
    combined.to_parquet(os.path.join(WEATHER_CACHE_PATH, 'hourly_weather_2018_2019.parquet'), index=False)
    print(f'Recovered: {len(retry_records)} airports')
    print(f'Total now: {combined["ORIGIN_AIRPORT"].nunique()}')
else:
    print('No new airports recovered')
print(f'Still failed: {len(still_failed)}')

In [110]:
weather_all = pd.read_parquet(os.path.join(WEATHER_CACHE_PATH, 'hourly_weather_2018_2019.parquet'))
print(f'Weather loaded: {weather_all.shape}')
print(f'Airports: {weather_all["ORIGIN_AIRPORT"].nunique()}')

Weather loaded: (5536320, 9)
Airports: 316


---
## 5 — Merge Weather with Flights

In [111]:
# Drop airports that failed weather fetch
airports_with_weather = weather_all['ORIGIN_AIRPORT'].unique()
before = len(df)
df = df[df['ORIGIN_AIRPORT'].isin(airports_with_weather)].copy()
print(f'Dropped {before - len(df):,} rows from airports without weather')

# Merge on airport + date + hour
weather_cols = ['DATE', 'HOUR', 'ORIGIN_AIRPORT', 'temp', 'precip', 'snowfall',
                'wind_speed', 'wind_gusts', 'weather_code']

df = df.merge(
    weather_all[weather_cols],
    on=['DATE', 'ORIGIN_AIRPORT', 'HOUR'],
    how='left'
)

print(f'\nRows after weather merge: {len(df):,}')
print(f'\nWeather nulls:')
wx_cols = ['temp', 'precip', 'snowfall', 'wind_speed', 'wind_gusts',
           'weather_code']
print(df[wx_cols].isnull().sum())

# Drop rows with missing weather (typically very few)
before = len(df)
df = df.dropna(subset=wx_cols)
print(f'\nDropped {before - len(df):,} rows with missing weather')
print(f'Final row count: {len(df):,}')

Dropped 0 rows from airports without weather

Rows after weather merge: 13,350,723

Weather nulls:
temp            0
precip          0
snowfall        0
wind_speed      0
wind_gusts      0
weather_code    0
dtype: int64

Dropped 0 rows with missing weather
Final row count: 13,350,723


In [112]:
# ---- Congestion (computed on full 13M before sampling) ----
print('Computing congestion on full dataset...')

orig_cong = df.groupby(['DATE', 'ORIGIN_AIRPORT', 'HOUR']).size().reset_index(name='ORIGIN_CONGESTION')
df = df.merge(orig_cong, on=['DATE', 'ORIGIN_AIRPORT', 'HOUR'], how='left')
del orig_cong

df['_ARR_HOUR'] = (df['SCHEDULED_ARRIVAL'] // 100).astype(int).clip(0, 23)
dest_cong = df.groupby(['DATE', 'DESTINATION_AIRPORT', '_ARR_HOUR']).size().reset_index(name='DEST_CONGESTION')
df = df.merge(dest_cong, on=['DATE', 'DESTINATION_AIRPORT', '_ARR_HOUR'], how='left')
df.drop(columns='_ARR_HOUR', inplace=True)
del dest_cong

print(f'ORIGIN_CONGESTION mean: {df["ORIGIN_CONGESTION"].mean():.1f}')
print(f'DEST_CONGESTION mean: {df["DEST_CONGESTION"].mean():.1f}')
corr_o = df['ORIGIN_CONGESTION'].corr(df['TARGET'])
corr_d = df['DEST_CONGESTION'].corr(df['TARGET'])
print(f'Corr with TARGET — origin: {corr_o:.4f}  dest: {corr_d:.4f}')

Computing congestion on full dataset...
ORIGIN_CONGESTION mean: 24.8
DEST_CONGESTION mean: 23.6
Corr with TARGET — origin: 0.0076  dest: -0.0274


In [113]:
# ---- Scheduled turnaround time (before sampling) ----
print('Computing scheduled turnaround time...')
df = df.sort_values(['TAIL_NUMBER', 'DATE', 'HOUR']).reset_index(drop=True)

# Previous flight's scheduled arrival (schedule data, not actual)
df['_PREV_SCHED_ARR'] = df.groupby('TAIL_NUMBER')['SCHEDULED_ARRIVAL'].shift(1)
df['_PREV_DATE'] = df.groupby('TAIL_NUMBER')['DATE'].shift(1)

# Convert HHMM to minutes-since-midnight for arithmetic
df['_curr_dep_min'] = (df['SCHEDULED_DEPARTURE'] // 100) * 60 + (df['SCHEDULED_DEPARTURE'] % 100)
df['_prev_arr_min'] = (df['_PREV_SCHED_ARR'] // 100) * 60 + (df['_PREV_SCHED_ARR'] % 100)

# Turnaround = current departure - previous arrival (in minutes)
# Only valid if same day (overnight turnarounds are long anyway)
same_day = df['DATE'] == df['_PREV_DATE']
df['TURNAROUND_MIN'] = np.where(same_day, df['_curr_dep_min'] - df['_prev_arr_min'], -1)

df.drop(columns=['_PREV_SCHED_ARR', '_PREV_DATE', '_curr_dep_min', '_prev_arr_min'], inplace=True)

valid = df[df['TURNAROUND_MIN'] > 0]
corr = valid['TURNAROUND_MIN'].corr(valid['TARGET'])
print(f'Valid turnarounds: {len(valid):,} ({len(valid)/len(df):.1%})')
print(f'Mean turnaround: {valid["TURNAROUND_MIN"].mean():.0f} min')
print(f'Corr with TARGET: {corr:.4f}')

Computing scheduled turnaround time...
Valid turnarounds: 10,075,268 (75.5%)
Mean turnaround: 71 min
Corr with TARGET: 0.0112


---
## 6 — Sample & Save
The full dataset may be several million rows. We sample strategically — enough to have dense interaction rates, small enough for Colab modelling.

**Important:** We sample AFTER the weather merge so every row has weather. We stratify by year×month to preserve temporal and seasonal balance.

In [114]:
df['YEAR'] = pd.to_datetime(df['DATE']).dt.year

In [ ]:
# Stratified sampling: 200k flights from 2018, 100k from 2019.
# Stratification is by year × month so each month is represented in proportion
# to its size (preventing the sample from skewing toward heavier travel months).
# random_state=42 makes the sample deterministic across reruns.
df['YEAR'] = df['DATE'].dt.year

samples = []
for year, target_n in [(2018, 200_000), (2019, 100_000)]:
    year_df = df[df['YEAR'] == year]
    frac = target_n / len(year_df)
    sampled = year_df.groupby(year_df['DATE'].dt.month, group_keys=False).apply(
        lambda x: x.sample(frac=frac, random_state=42)
    )
    samples.append(sampled)

# Re-sort chronologically so downstream temporal features (rolling delays,
# previous-leg lookups) compute correctly.
df = pd.concat(samples, ignore_index=True)
df['YEAR'] = df['DATE'].dt.year
df = df.sort_values(['DATE', 'SCHEDULED_DEPARTURE']).reset_index(drop=True)

print(f'Total: {len(df):,}')
print(df['YEAR'].value_counts().sort_index())
print(f'\nClass balance:')
print(df['TARGET'].value_counts(normalize=True).round(3))

In [116]:
# ---- Final column inventory ----
print(f'Columns ({len(df.columns)}):')
for col in df.columns:
    null_pct = df[col].isnull().mean() * 100
    dtype = df[col].dtype
    print(f'  {col:<30} {str(dtype):<15} nulls={null_pct:.1f}%')

Columns (26):
  DATE                           datetime64[us]  nulls=0.0%
  AIRLINE                        str             nulls=0.0%
  ORIGIN_AIRPORT                 str             nulls=0.0%
  DESTINATION_AIRPORT            str             nulls=0.0%
  SCHEDULED_DEPARTURE            int64           nulls=0.0%
  DEPARTURE_DELAY                float64         nulls=0.0%
  DISTANCE                       float64         nulls=0.0%
  TAIL_NUMBER                    str             nulls=0.0%
  SCHEDULED_ARRIVAL              int64           nulls=0.0%
  SCHEDULED_ELAPSED_TIME         float64         nulls=0.0%
  DAY_OF_WEEK                    int64           nulls=0.0%
  ArrDelay                       float64         nulls=0.0%
  YEAR                           int32           nulls=0.0%
  HOUR                           int64           nulls=0.0%
  TARGET                         int64           nulls=0.0%
  LATITUDE                       float64         nulls=0.0%
  LONGITUDE               

In [117]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'Saved to {OUTPUT_PATH}')

Shape: (300000, 26)
Columns: ['DATE', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_DELAY', 'DISTANCE', 'TAIL_NUMBER', 'SCHEDULED_ARRIVAL', 'SCHEDULED_ELAPSED_TIME', 'DAY_OF_WEEK', 'ArrDelay', 'YEAR', 'HOUR', 'TARGET', 'LATITUDE', 'LONGITUDE', 'temp', 'precip', 'snowfall', 'wind_speed', 'wind_gusts', 'weather_code', 'ORIGIN_CONGESTION', 'DEST_CONGESTION', 'TURNAROUND_MIN']
Saved to C:\YEAR 3 - FIRST SEM\flight-delay-predictor-6\Data\01_cleaned_data.csv


In [118]:
df = df.drop(columns=['LATITUDE', 'LONGITUDE'])
df.to_csv(OUTPUT_PATH, index=False)
print(f'Final: {df.shape}')
print(df.columns.tolist())

Final: (300000, 24)
['DATE', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_DELAY', 'DISTANCE', 'TAIL_NUMBER', 'SCHEDULED_ARRIVAL', 'SCHEDULED_ELAPSED_TIME', 'DAY_OF_WEEK', 'ArrDelay', 'YEAR', 'HOUR', 'TARGET', 'temp', 'precip', 'snowfall', 'wind_speed', 'wind_gusts', 'weather_code', 'ORIGIN_CONGESTION', 'DEST_CONGESTION', 'TURNAROUND_MIN']
